# Turning Treasury Yields Into Supervised Learning Windows

Phase 1 gave us clean daily Treasury yields. Phase 2 helped us look at the data like analysts. Phase 3 turns the time series into supervised-learning examples that a sequence model can eventually consume.

We still are **not training a model** here. We are defining the learning problem carefully.

## Why this is supervised learning

A raw time series is just observations ordered by date. Supervised learning needs examples with inputs and answers.

For each forecast origin date `t`:

- `X` is the information available up to `t`.
- `y` is what happens after `t`.

That pairing is what makes the dataset supervised. The model will later learn a mapping from recent yield-curve history to future yield changes.

## The concrete example

With `lookback = 60` and `horizon = 1`:

- `X` is the previous 60 trading days of features ending at date `t`.
- `y` is the next trading day's seven yield changes, from `t` to `t + 1`.

For each maturity we create four features: level, 1-day change, 5-day change, and 21-day change. There are seven maturities, so each day has `7 * 4 = 28` features.

Final tensor shapes:

- one input example: `(60, 28)`
- one target example: `(7,)`
- full input tensor: `(num_examples, 60, 28)`
- full target tensor: `(num_examples, 7)`

In [ ]:
# ruff: noqa: E402, I001
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd

from market_resonance.features import (
    build_supervised_windows,
    chronological_train_validation_test_split,
    create_treasury_features,
    standardize_splits,
)

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "treasury_yields_daily.csv"

In [ ]:
yields = pd.read_csv(DATA_PATH, parse_dates=["date"])
yields.head()

## Feature table

The feature table stays aligned by date. The early rows have missing trailing-change values because a 21-day change cannot exist until we have at least 21 earlier trading observations. The window builder starts only once all required features are available.

In [ ]:
features = create_treasury_features(yields)
features.filter(regex="date|10Y").head(25).tail()

## Build supervised windows

The reusable Python module creates batch-first arrays. The feature tensor is ready for sequence models such as an LSTM, but no model is trained in this phase.

In [ ]:
windows = build_supervised_windows(yields, lookback=60, horizon=1)
{
    "X_shape": windows.X.shape,
    "y_shape": windows.y.shape,
    "num_features": len(windows.feature_columns),
    "num_targets": len(windows.target_columns),
}

## Inspect one `X` and `y` pair

The first example below has 60 rows of input history. Its target date is the next trading row after the input window ends.

In [ ]:
example_index = 0
example_X = windows.X[example_index]
example_y = windows.y[example_index]

{
    "sample_start": windows.sample_start_dates.iloc[example_index].date(),
    "sample_end": windows.sample_end_dates.iloc[example_index].date(),
    "target_date": windows.target_dates.iloc[example_index].date(),
    "X_shape": example_X.shape,
    "y_shape": example_y.shape,
}

In [ ]:
pd.DataFrame(example_X, columns=windows.feature_columns).tail()

In [ ]:
pd.Series(example_y, index=windows.target_columns)

## Chronological train/validation/test split

Time series examples must stay in chronological order. Random shuffling would mix future regimes into training and make validation or test results too optimistic.

In [ ]:
splits = chronological_train_validation_test_split(windows)
{
    "train": splits.train.X.shape,
    "validation": splits.validation.X.shape,
    "test": splits.test.X.shape,
    "train_end": splits.train.sample_end_dates.iloc[-1].date(),
    "validation_end": splits.validation.sample_end_dates.iloc[-1].date(),
    "test_end": splits.test.sample_end_dates.iloc[-1].date(),
}

## Normalize without leakage

Normalization is fitted on `train.X` only. The frozen mean and scale are then applied to validation and test. This prevents validation/test distribution information from leaking backward into training.

In [ ]:
standardized_splits, standardizer = standardize_splits(splits)
{
    "standardized_train_X_shape": standardized_splits.train.X.shape,
    "mean_shape": standardizer.mean_.shape,
    "scale_shape": standardizer.scale_.shape,
    "train_mean_after_standardization": standardized_splits.train.X.mean().round(6),
    "train_std_after_standardization": standardized_splits.train.X.std().round(6),
}

## Phase 3 stopping point

At this point we have supervised arrays, chronological splits, and train-only normalization. That is the data interface a model can use later. We deliberately stop before baselines or neural networks.